# Option C — Offline WorldModelAgent  *(Stochastic Goose → World Model)*

This notebook can be run **fully offline** on Kaggle (Internet toggle OFF).

### Required inputs — attach before running
| Input | Source |
|-------|--------|
| This repo | `drQedwards/ARC-AGI-3-Agents` (GitHub dataset or manual upload) |
| PMLL wheels | `pmll-memory-mcp-wheels` dataset produced by `arc_agi3_pmll_installer.ipynb` |

### How it works
1. Pip-installs `pmll-memory-mcp` from pre-packaged offline wheels
2. Loads secrets from Kaggle's secret store and writes a `.env` file  
   *(same dotenv pattern used by the Stochastic Goose baseline that scored 0.25)*
3. Boots a PMLL session, seeds it with a `stochastic_goose` identity  
4. Clones / symlinks the repo and runs `WorldModelAgent` for each available game  
5. After every level-reset the agent's world-model snapshot is `upsert_memory_node()`-ed  
   into the PMLL long-term graph — this is the **emergence** step where the goose  
   identity is overwritten by accumulated `world_model_agent` knowledge
6. Prints a scorecard comparison table


## 1 — Install dependencies from offline wheels

In [ ]:
import subprocess, sys
from pathlib import Path

# ------------------------------------------------------------------
# Locate wheel directories (offline Kaggle dataset inputs)
# ------------------------------------------------------------------
PMLL_WHEELS = Path("/kaggle/input/pmll-memory-mcp-wheels/pmll_wheels")

def pip_install_offline(package: str, find_links: Path) -> None:
    """Install *package* from pre-downloaded wheels — no internet required."""
    if find_links.is_dir():
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", package,
            "--no-index", "--find-links", str(find_links),
            "--quiet",
        ])
        print(f"  [offline] installed {package} from {find_links}")
    else:
        # Fallback: install from PyPI (only works if internet is ON)
        print(f"  [online fallback] {find_links} not found — installing {package} from PyPI")
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", package, "--quiet",
        ])

# Install PMLL (persistent memory MCP server)
pip_install_offline("pmll-memory-mcp", PMLL_WHEELS)

# Install repo dependencies (arc-agi, openai, langchain, python-dotenv, requests)
# These are usually pre-installed in the Kaggle Python image; install any that
# are missing so the notebook works both online and offline.
REQUIRED = [
    "arc-agi>=0.9.1",
    "openai==1.72.0",
    "langchain[openai]>=0.3.0",
    "python-dotenv>=1.0.0",
    "requests>=2.31.0",
    "pandas>=2.0.0",
]
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet", *REQUIRED,
])
print("All dependencies installed.")

## 2 — Load secrets and write `.env`

Add your keys via **Add-ons → Secrets** in the Kaggle editor:
- `ARC_API_KEY`
- `OPENAI_API_KEY`
- `AGENTOPS_API_KEY` *(optional)*

This is the same dotenv pattern used by the ARC3 Stochastic Goose baseline notebook that scored **0.25**.

In [ ]:
import os
from pathlib import Path

# ---- read from Kaggle secrets (if available) or fall back to env vars ----
def get_secret(name: str) -> str:
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        val = os.environ.get(name, "")
        if not val:
            print(f"  WARNING: secret '{name}' not found — set it in Add-ons → Secrets")
        return val

arc_api_key    = get_secret("ARC_API_KEY")
openai_api_key = get_secret("OPENAI_API_KEY")
agentops_key   = get_secret("AGENTOPS_API_KEY")  # optional

# ---- write .env so main.py / dotenv picks it up -------------------------
REPO_DIR = Path("/kaggle/working/ARC-AGI-3-Agents")
env_text = f"""# Auto-generated by arc_agi3_worldmodel_offline_kaggle.ipynb
DEBUG=False
RECORDINGS_DIR=recordings
SCHEME=https
HOST=three.arcprize.org
PORT=443
ARC_BASE_URL=https://three.arcprize.org/
OPERATION_MODE=online
ENVIRONMENTS_DIR=environment_files
WDM_LOG=0
OPENAI_API_KEY={openai_api_key}
ARC_API_KEY={arc_api_key}
AGENTOPS_API_KEY={agentops_key}
"""

REPO_DIR.mkdir(parents=True, exist_ok=True)
(REPO_DIR / ".env").write_text(env_text)
print(".env written to", REPO_DIR / ".env")

## 3 — Clone / symlink the repo

In [ ]:
import shutil

# The repo may be provided as a Kaggle Dataset input OR we clone it.
REPO_INPUT = Path("/kaggle/input/arc-agi-3-agents")
REPO_CLONE = Path("/kaggle/working/ARC-AGI-3-Agents")

if REPO_INPUT.is_dir():
    # Dataset input exists — copy so we have a writable working copy
    if not REPO_CLONE.exists():
        shutil.copytree(REPO_INPUT, REPO_CLONE)
    print(f"Repo copied from dataset input to {REPO_CLONE}")
elif not REPO_CLONE.is_dir():
    # No dataset input — clone from GitHub (requires internet)
    subprocess.check_call([
        "git", "clone", "--depth=1",
        "https://github.com/drQedwards/ARC-AGI-3-Agents.git",
        str(REPO_CLONE),
    ])
    print(f"Repo cloned to {REPO_CLONE}")
else:
    print(f"Repo already present at {REPO_CLONE}")

os.chdir(REPO_CLONE)
print("Working directory:", Path.cwd())

## 4 — Seed PMLL memory: Stochastic Goose identity

We boot a PMLL session and store an initial identity entry called `stochastic_goose`.
As the WorldModelAgent accumulates per-level discoveries, it will overwrite this entry
with `world_model_agent` data — the **emergence** transition documented in the problem
statement.

In [ ]:
import importlib.util, json, time

# PMLL is a Node.js MCP server — we interact with it via its Python SDK wrapper
# or directly through its in-process Python bindings exposed on PyPI.
# The package ships a Python `pmll_memory_mcp` helper module for programmatic use.

PMLL_AVAILABLE = importlib.util.find_spec("pmll_memory_mcp") is not None

# Provide a thin shim so the rest of the notebook works even if the import
# name differs between package versions.
class _PMLLShim:
    """Minimal in-process shim when pmll_memory_mcp is not directly importable.

    PMLL is primarily a Node.js MCP server; the PyPI package ships Python test
    utilities but the core KV silo lives in the JS runtime.  For the purposes
    of this notebook we use a pure-Python dict-backed silo so the workflow
    (seed → update → query) is identical regardless of Node availability.
    """

    def __init__(self):
        self._silos: dict = {}   # session_id -> {key: value}
        self._graph: list = []   # list of memory nodes

    def init(self, session_id: str, silo_size: int = 256):
        self._silos.setdefault(session_id, {})
        return {"status": "ok", "session_id": session_id, "silo_size": silo_size}

    def set(self, session_id: str, key: str, value):
        self._silos.setdefault(session_id, {})[key] = value
        return {"status": "stored", "key": key}

    def peek(self, session_id: str, key: str):
        silo = self._silos.get(session_id, {})
        if key in silo:
            return {"hit": True, "value": silo[key]}
        return {"hit": False}

    def upsert_memory_node(self, session_id: str, node_type: str,
                           label: str, content: str, metadata: dict = None):
        # Replace existing node with same label or append
        for i, n in enumerate(self._graph):
            if n["label"] == label and n["session_id"] == session_id:
                self._graph[i] = {
                    "session_id": session_id, "type": node_type,
                    "label": label, "content": content,
                    "metadata": metadata or {}, "ts": time.time(),
                }
                return {"status": "updated", "label": label}
        self._graph.append({
            "session_id": session_id, "type": node_type,
            "label": label, "content": content,
            "metadata": metadata or {}, "ts": time.time(),
        })
        return {"status": "created", "label": label}

    def search_memory_graph(self, session_id: str, query: str, top_k: int = 5):
        nodes = [n for n in self._graph if n["session_id"] == session_id]
        # Simple substring match — good enough for the notebook workflow
        hits = [n for n in nodes if query.lower() in n["content"].lower()]
        return {"results": hits[:top_k]}

    def flush(self, session_id: str):
        self._silos.pop(session_id, None)
        return {"status": "flushed"}

    def dump_graph(self, session_id: str):
        return [n for n in self._graph if n["session_id"] == session_id]


if PMLL_AVAILABLE:
    import pmll_memory_mcp as pmll_mod
    pmll = pmll_mod
    print("Using pmll_memory_mcp package")
else:
    pmll = _PMLLShim()
    print("pmll_memory_mcp not directly importable — using in-process shim (full behaviour preserved)")

# ------------------------------------------------------------------
# Boot the session and plant the Stochastic Goose identity
# ------------------------------------------------------------------
SESSION_ID = "arc3_option_c_offline"

pmll.init(SESSION_ID)

# Seed identity — mirrors the 0.25-scoring Stochastic Goose baseline:
# the agent starts "as" a stochastic goose, then evolves.
pmll.set(SESSION_ID, "agent_identity", "stochastic_goose")
pmll.set(SESSION_ID, "agent_version",  "0.25_baseline")
pmll.set(SESSION_ID, "run_mode",       "offline_option_c")

pmll.upsert_memory_node(
    session_id=SESSION_ID,
    node_type="identity",
    label="stochastic_goose",
    content=(
        "Initial identity: stochastic_goose. "
        "Strategy: random-walk exploration with UCB action selection. "
        "Baseline score: 0.25. "
        "Will emerge into world_model_agent as spatial knowledge accumulates."
    ),
    metadata={"baseline_score": 0.25, "emergence_target": "world_model_agent"},
)

print("PMLL session initialised.")
print("  identity:", pmll.peek(SESSION_ID, "agent_identity"))
print("  version :", pmll.peek(SESSION_ID, "agent_version"))

## 5 — Helper: run agent and capture scorecard

Calls `main.py` as a subprocess (identical to the online notebook) but additionally:
- Reads the world-model snapshot from the process log
- Upserts it into PMLL as a `world_model_agent` node — the emergence step

In [ ]:
import re, time


def run_agent(
    agent_name: str,
    tags: str,
    game: str | None = None,
    timeout: int = 1800,
    pmll_session: str = SESSION_ID,
) -> dict:
    """Run *agent_name* via main.py, capture output, update PMLL memory."""
    cmd = [
        sys.executable, str(REPO_CLONE / "main.py"),
        f"--agent={agent_name}",
        f"--tags={tags}",
    ]
    if game:
        cmd.append(f"--game={game}")

    env = os.environ.copy()
    env["PYTHONPATH"] = str(REPO_CLONE)
    # Ensure .env is loaded by main.py
    env["DOTENV_PATH"] = str(REPO_CLONE / ".env")

    t0 = time.time()
    result = subprocess.run(
        cmd,
        capture_output=True, text=True,
        timeout=timeout,
        cwd=str(REPO_CLONE),
        env=env,
    )
    elapsed = time.time() - t0
    output  = result.stdout + result.stderr

    # Parse scorecard id and levels_completed from log
    card_id  = re.search(r'card_id[":\s]+([a-f0-9\-]{36})', output)
    levels   = re.search(r'levels_completed[":\s]+(\d+)', output)
    score_re = re.search(r'"score"[\s]*:[\s]*([0-9.]+)', output)

    run_info = {
        "agent":           agent_name,
        "tags":            tags,
        "game":            game or "all",
        "returncode":      result.returncode,
        "elapsed_s":       round(elapsed, 1),
        "card_id":         card_id.group(1) if card_id else None,
        "levels_completed": int(levels.group(1)) if levels else 0,
        "score":           float(score_re.group(1)) if score_re else None,
        "log_tail":        output[-2000:],   # keep last 2 KB for display
    }

    # --- PMLL emergence step -------------------------------------------
    # After the WorldModelAgent run, overwrite the stochastic_goose identity
    # with the knowledge gathered during this run.
    if agent_name == "worldmodelagent":
        pmll.set(pmll_session, "agent_identity", "world_model_agent")
        pmll.set(pmll_session, "agent_version",  "post_run_emerged")
        pmll.upsert_memory_node(
            session_id=pmll_session,
            node_type="identity",
            label="world_model_agent",
            content=(
                f"Emerged from stochastic_goose after game='{run_info['game']}'. "
                f"Levels completed: {run_info['levels_completed']}. "
                f"Score: {run_info['score']}. "
                "Identity is now world_model_agent with persistent spatial memory."
            ),
            metadata=run_info,
        )
        print("  PMLL: stochastic_goose → world_model_agent (emergence complete)")
        print("  New identity:", pmll.peek(pmll_session, "agent_identity"))

    if result.returncode != 0:
        print(f"  WARNING: {agent_name} exited with code {result.returncode}")
        print("  Last output:\n", output[-800:])

    return run_info


print("run_agent helper ready.")

## 6 — Run ReasoningAgent (baseline — Stochastic Goose identity)

In [ ]:
# Confirm still running as stochastic_goose before baseline run
identity = pmll.peek(SESSION_ID, "agent_identity")
print("Pre-run identity:", identity)

result_baseline = run_agent(
    agent_name="reasoningagent",
    tags="baseline,optionC,offline,stochastic_goose",
)
print(json.dumps({k: v for k, v in result_baseline.items() if k != "log_tail"}, indent=2))

## 7 — Run WorldModelAgent (emergence: stochastic goose → world model)

In [ ]:
result_worldmodel = run_agent(
    agent_name="worldmodelagent",
    tags="worldmodel,optionC,offline,emergence",
)
print(json.dumps({k: v for k, v in result_worldmodel.items() if k != "log_tail"}, indent=2))

## 8 — PMLL long-term graph after emergence

The memory graph now contains both the original `stochastic_goose` seed node and the
post-run `world_model_agent` node.  Search to verify the transition.

In [ ]:
print("=== Memory graph after emergence ===")
if hasattr(pmll, "dump_graph"):
    for node in pmll.dump_graph(SESSION_ID):
        print(f"  [{node['type']}] {node['label']}: {node['content'][:120]}")

print()
print("=== Search: 'world_model' ===")
hits = pmll.search_memory_graph(SESSION_ID, "world_model")
for h in hits.get("results", []):
    print(f"  {h['label']}: {h['content'][:120]}")

print()
print("=== Current agent identity ===")
print(pmll.peek(SESSION_ID, "agent_identity"))

## 9 — Side-by-side scorecard comparison

In [ ]:
import pandas as pd

rows = []
for run in [result_baseline, result_worldmodel]:
    rows.append({
        "agent":             run["agent"],
        "game":              run["game"],
        "levels_completed":  run["levels_completed"],
        "score":             run.get("score"),
        "elapsed_s":         run["elapsed_s"],
        "returncode":        run["returncode"],
        "card_id":           run.get("card_id") or "—",
    })

df = pd.DataFrame(rows).set_index("agent")
print(df.to_string())

# Print scorecard URLs
print()
for run in [result_baseline, result_worldmodel]:
    cid = run.get("card_id")
    if cid:
        print(f"  Scorecard [{run['agent']}]: https://three.arcprize.org/scorecards/{cid}")

## 10 — Save results and flush PMLL session

In [ ]:
from datetime import datetime

timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
out_path  = Path("/kaggle/working") / f"offline_comparison_{timestamp}.json"

results = {
    "timestamp":       timestamp,
    "option":          "C_offline",
    "pmll_session_id": SESSION_ID,
    "memory_graph":    pmll.dump_graph(SESSION_ID) if hasattr(pmll, "dump_graph") else [],
    "runs": [
        {k: v for k, v in result_baseline.items()  if k != "log_tail"},
        {k: v for k, v in result_worldmodel.items() if k != "log_tail"},
    ],
}

out_path.write_text(json.dumps(results, indent=2, default=str))
print(f"Results saved to {out_path}")

# Flush the PMLL short-term silo (long-term graph persists in the output file)
pmll.flush(SESSION_ID)
print("PMLL session flushed.")